# Exact illustrations for release-conditional PII redaction claims

**All scenarios in this notebook are analytical constructions. They are not real-world measurements, model evaluations, dataset benchmarks, or estimates of actual deployment risk.** The figures demonstrate why mention-level metrics, release conditioning, audit sample sizes, and candidate coverage answer different questions.

This notebook accompanies *Position: PII Redaction Claims Must Specify the Release Mechanism*. It runs on CPU, does not download data or models, and contains no random experiment. Dependencies are Python, NumPy, Matplotlib, and SciPy. Outputs are written to `illustration_outputs/` in the current working directory. Tested dependency versions and deterministic numerical assertions are recorded in `environment.json` and `results.json`.

The audited event is **at least one policy-sensitive item remains visible in a released record**. Policy-sensitive items, release decisions, and visibility are specified by each construction. These examples concern content removal and do not assert protection against every possible inference from a released record.

In [1]:
from pathlib import Path
import csv
import json
import math
import platform
import sys
import numpy as np
import scipy
from scipy.stats import beta as beta_distribution
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT = Path("illustration_outputs")
OUT.mkdir(parents=True, exist_ok=True)
VERSIONS = {"python": platform.python_version(), "numpy": np.__version__,
            "scipy": scipy.__version__, "matplotlib": matplotlib.__version__}
(OUT / "environment.json").write_text(json.dumps(VERSIONS, indent=2) + "\n")
print("Dependency versions:", json.dumps(VERSIONS, sort_keys=True))

plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 8,
    "axes.titlesize": 8.5, "axes.labelsize": 8,
    "xtick.labelsize": 7.5, "ytick.labelsize": 7.5,
    "legend.fontsize": 7, "pdf.fonttype": 42, "ps.fonttype": 42,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.linewidth": 0.6, "grid.linewidth": 0.4,
    "savefig.dpi": 240,
})
BLUE, ORANGE, GREEN = "#2166AC", "#C55A11", "#3B7B57"

def write_csv(name, rows):
    assert rows and isinstance(rows[0], dict)
    with (OUT / name).open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


Dependency versions: {"matplotlib": "3.10.8", "numpy": "2.3.5", "python": "3.12.14", "scipy": "1.17.0"}


## 1. Identical mention scores, different record-level harm

Each of 1,000 records contains exactly 100 distinct policy-sensitive mentions. Every record is released; an unremoved mention makes a record bad. System A misses one mention in every record. System B misses all 100 mentions in ten records and no mentions elsewhere. There are no false positive redactions. Both systems therefore have exactly the same micro precision, recall, $F_1$, and $F_{10}$, while their record-level bad-release rates differ by a factor of 100.

For mention counts $TP,FP,FN$, the score is
\[
F_\beta=\frac{(1+\beta^2)TP}{(1+\beta^2)TP+\beta^2FN+FP}.
\]
This construction does not claim that concentrating misses is generally desirable. It shows that a mention metric cannot determine the probability of a bad record. Severity, which mentions are missed, and who is affected remain additional questions.

In [2]:
N_RECORDS = 1000
MENTIONS_PER_RECORD = 100
misses_A = np.ones(N_RECORDS, dtype=int)
misses_B = np.concatenate([np.full(10, MENTIONS_PER_RECORD, dtype=int),
                           np.zeros(N_RECORDS - 10, dtype=int)])

metric_rows = []
for name, misses in [("A: dispersed misses", misses_A), ("B: concentrated misses", misses_B)]:
    fn = int(misses.sum())
    tp = N_RECORDS * MENTIONS_PER_RECORD - fn
    fp = 0
    fscore = lambda b: (1 + b*b) * tp / ((1 + b*b) * tp + b*b*fn + fp)
    metric_rows.append({
        "system": name, "records": N_RECORDS,
        "mentions_per_record": MENTIONS_PER_RECORD,
        "true_positive_mentions": tp, "false_positive_mentions": fp,
        "false_negative_mentions": fn, "micro_precision": tp/(tp+fp),
        "micro_recall": tp/(tp+fn), "micro_F1": fscore(1), "micro_F10": fscore(10),
        "bad_released_records": int((misses > 0).sum()),
        "released_records": N_RECORDS,
        "bad_release_rate": float((misses > 0).mean()),
    })
for key in ["micro_precision", "micro_recall", "micro_F1", "micro_F10"]:
    assert metric_rows[0][key] == metric_rows[1][key]
assert metric_rows[0]["micro_recall"] == 0.99
assert metric_rows[0]["bad_release_rate"] == 1
assert metric_rows[1]["bad_release_rate"] == 0.01
assert metric_rows[0]["false_negative_mentions"] == metric_rows[1]["false_negative_mentions"] == 1000
write_csv("mention_vs_record.csv", metric_rows)
for row in metric_rows:
    print(row)


{'system': 'A: dispersed misses', 'records': 1000, 'mentions_per_record': 100, 'true_positive_mentions': 99000, 'false_positive_mentions': 0, 'false_negative_mentions': 1000, 'micro_precision': 1.0, 'micro_recall': 0.99, 'micro_F1': 0.9949748743718593, 'micro_F10': 0.9900980295078721, 'bad_released_records': 1000, 'released_records': 1000, 'bad_release_rate': 1.0}
{'system': 'B: concentrated misses', 'records': 1000, 'mentions_per_record': 100, 'true_positive_mentions': 99000, 'false_positive_mentions': 0, 'false_negative_mentions': 1000, 'micro_precision': 1.0, 'micro_recall': 0.99, 'micro_F1': 0.9949748743718593, 'micro_F10': 0.9900980295078721, 'bad_released_records': 10, 'released_records': 1000, 'bad_release_rate': 0.01}


## 2. Release conditioning changes the denominator

Define $Y=1$ when a processed record would be unsafe under the stated visibility criterion, and $G=1$ when the gate releases it. Consider a fixed constructed population of 100,000 processed records. The gate releases 10,000, of which 1,000 are unsafe. The gate withholds 90,000, all of which are safe in this construction. Thus the joint frequency $\Pr(Y=1,G=1)$ is 1%, whereas the risk among released records $\Pr(Y=1\mid G=1)$ is 10%. Here the pre-gate unsafe frequency $\Pr(Y=1)$ happens to equal the joint frequency because no unsafe record is withheld.

A second exact table varies how many of the same 1,000 unsafe records are released while keeping the total release count at 10,000. At 100 unsafe releases, $Y$ and $G$ are independent in the finite population. This is a population construction, not a proposed gate or an empirical estimate.

In [3]:
POPULATION = 100_000
RELEASED = 10_000
TOTAL_BAD = 1_000
population_rows = [
    {"gate": "released", "unsafe_records": 1_000, "safe_records": 9_000, "total_records": 10_000},
    {"gate": "withheld", "unsafe_records": 0, "safe_records": 90_000, "total_records": 90_000},
]
release_illustration = {
    "processed_records": POPULATION, "released_records": RELEASED,
    "unsafe_released_records": TOTAL_BAD,
    "unsafe_withheld_records": 0,
    "coverage": RELEASED/POPULATION,
    "pre_gate_unsafe_rate": TOTAL_BAD/POPULATION,
    "joint_bad_and_released_rate": TOTAL_BAD/POPULATION,
    "conditional_bad_release_rate": TOTAL_BAD/RELEASED,
}
assert release_illustration["joint_bad_and_released_rate"] == 0.01
assert release_illustration["conditional_bad_release_rate"] == 0.10
assert release_illustration["coverage"] == 0.10
write_csv("release_population.csv", population_rows)

selection_rows = []
for bad_released in [0, 50, 100, 250, 500, 750, 1000]:
    bad_withheld = TOTAL_BAD - bad_released
    safe_released = RELEASED - bad_released
    safe_withheld = POPULATION - RELEASED - bad_withheld
    assert min(bad_released, bad_withheld, safe_released, safe_withheld) >= 0
    # Pearson correlation for binary Y and G in this fixed population.
    p_y = TOTAL_BAD/POPULATION
    p_g = RELEASED/POPULATION
    p_yg = bad_released/POPULATION
    corr = (p_yg-p_y*p_g)/math.sqrt(p_y*(1-p_y)*p_g*(1-p_g))
    selection_rows.append({
        "bad_released": bad_released, "bad_withheld": bad_withheld,
        "safe_released": safe_released, "safe_withheld": safe_withheld,
        "release_coverage": p_g, "pre_gate_bad_rate": p_y,
        "joint_bad_and_released_rate": p_yg,
        "conditional_bad_release_rate": bad_released/RELEASED,
        "binary_gate_error_correlation": corr,
    })
assert math.isclose(selection_rows[2]["binary_gate_error_correlation"], 0, abs_tol=1e-15)
write_csv("release_selection_sweep.csv", selection_rows)
print(json.dumps(release_illustration, indent=2))


{
  "processed_records": 100000,
  "released_records": 10000,
  "unsafe_released_records": 1000,
  "unsafe_withheld_records": 0,
  "coverage": 0.1,
  "pre_gate_unsafe_rate": 0.01,
  "joint_bad_and_released_rate": 0.01,
  "conditional_bad_release_rate": 0.1
}


### 2a. A stricter gate can increase conditional risk

Use the same fixed processed outputs for 1,000 records, exactly eight of which are unsafe. The old gate releases 800 records, including all eight unsafe records. The new gate releases a subset of 80 of those 800, still including all eight unsafe records. The joint bad-release frequency stays at 0.8%, but the conditional bad-release rate rises from 1% to 10% as coverage decreases from 80% to 8%. No output has become less safe; only the gate has changed.

For fixed marginal unsafe frequency $q=\Pr(Y=1)$ and release coverage $c=\Pr(G=1)>0$, the feasible bounds are
\[
\max\{0,(q+c-1)/c\}\leq\Pr(Y=1\mid G=1)\leq\min\{1,q/c\}.
\]
They follow from the intersection bounds $\max\{0,q+c-1\}\leq\Pr(Y=1,G=1)\leq\min\{q,c\}$. Here $q$ is a **pre-gate marginal**, not the joint bad-release frequency. The bounds are achievable by placing as much or as little unsafe mass as possible among releases; at finite sample sizes, counts must also be integers.

In [4]:
def conditional_risk_bounds(q, coverage):
    assert 0 <= q <= 1 and 0 < coverage <= 1
    return max(0.0, (q + coverage - 1)/coverage), min(1.0, q/coverage)

# Explicit record identities make the nested-gate assumption verifiable.
unsafe_ids = set(range(8))
old_release_ids = set(range(800))
new_release_ids = set(range(80))
assert new_release_ids < old_release_ids
assert unsafe_ids <= new_release_ids

gate_tightening_rows = []
for name, released_ids in [("old", old_release_ids), ("stricter_nested", new_release_ids)]:
    bad_released = len(unsafe_ids & released_ids)
    c = len(released_ids)/1000
    q = len(unsafe_ids)/1000
    lo, hi = conditional_risk_bounds(q, c)
    r = bad_released/len(released_ids)
    assert lo <= r <= hi
    gate_tightening_rows.append({
        "gate": name, "processed_records": 1000,
        "released_records": len(released_ids), "unsafe_released_records": bad_released,
        "pre_gate_unsafe_rate_q": q, "coverage_c": c,
        "joint_bad_and_released_rate": bad_released/1000,
        "conditional_bad_release_rate": r,
        "feasible_conditional_lower_bound": lo,
        "feasible_conditional_upper_bound": hi,
    })
assert gate_tightening_rows[0]["conditional_bad_release_rate"] == 0.01
assert gate_tightening_rows[1]["conditional_bad_release_rate"] == 0.1
assert gate_tightening_rows[0]["joint_bad_and_released_rate"] == gate_tightening_rows[1]["joint_bad_and_released_rate"] == 0.008
write_csv("gate_tightening.csv", gate_tightening_rows)
for row in gate_tightening_rows:
    print(row)


{'gate': 'old', 'processed_records': 1000, 'released_records': 800, 'unsafe_released_records': 8, 'pre_gate_unsafe_rate_q': 0.008, 'coverage_c': 0.8, 'joint_bad_and_released_rate': 0.008, 'conditional_bad_release_rate': 0.01, 'feasible_conditional_lower_bound': 0.0, 'feasible_conditional_upper_bound': 0.01}
{'gate': 'stricter_nested', 'processed_records': 1000, 'released_records': 80, 'unsafe_released_records': 8, 'pre_gate_unsafe_rate_q': 0.008, 'coverage_c': 0.08, 'joint_bad_and_released_rate': 0.008, 'conditional_bad_release_rate': 0.1, 'feasible_conditional_lower_bound': 0.0, 'feasible_conditional_upper_bound': 0.1}


### 2b. Records without protected content can dilute the denominator

In a fixed population of 1,000 records, 100 contain policy-required content and ten of those fail the removal policy. The other 900 have no policy-required content and therefore cannot fail this particular containment loss. Initially, only the 100 protected-content records are released. A revised gate additionally releases all 900 other records. No record, mask, policy, or protected-content release changes.

The failure proportion among all released records falls from 10% to 1%, while failure among released records containing protected content remains 10%. With $H=\mathbf{1}\{S_\pi(X)\ne\varnothing\}$ and the containment loss $L=\mathbf{1}\{S_\pi(X)\not\subseteq R(X)\}$, $L=1$ implies $H=1$. Consequently,
\[
r=\Pr(H=1\mid G=1)r_+,\qquad r_+=\Pr(L=1\mid G=1,H=1).
\]
This is an identity for this policy-defined loss. It does not imply that an apparently empty record is safe against every inference attack. $H$ is determined from the specified policy and complete audit labels, not from whether the detector emitted a candidate. Reporting $r_+$ requires reporting its own released-unit denominator.

In [5]:
protected_ids = set(range(100))
dilution_failure_ids = set(range(10))
dilution_rows = []
for name, released in [("protected_only", set(range(100))),
                       ("plus_900_without_protected_content", set(range(1000)))]:
    n = len(released)
    positive_n = len(released & protected_ids)
    k = len(released & dilution_failure_ids)
    r = k/n
    r_positive = k/positive_n
    prevalence = positive_n/n
    assert dilution_failure_ids <= protected_ids
    assert math.isclose(r, prevalence*r_positive)
    dilution_rows.append({
        "gate": name, "input_records": 1000, "released_records": n,
        "protected_content_released_records": positive_n,
        "failed_releases": k, "coverage": n/1000,
        "protected_content_prevalence_among_releases": prevalence,
        "all_release_failure_proportion": r,
        "protected_content_release_failure_proportion": r_positive,
        "joint_failed_release_proportion": k/1000,
    })
assert dilution_rows[0]["all_release_failure_proportion"] == .1
assert dilution_rows[1]["all_release_failure_proportion"] == .01
assert all(row["protected_content_release_failure_proportion"] == .1 for row in dilution_rows)
write_csv("protected_content_denominator.csv", dilution_rows)
print(json.dumps(dilution_rows, indent=2))


[
  {
    "gate": "protected_only",
    "input_records": 1000,
    "released_records": 100,
    "protected_content_released_records": 100,
    "failed_releases": 10,
    "coverage": 0.1,
    "protected_content_prevalence_among_releases": 1.0,
    "all_release_failure_proportion": 0.1,
    "protected_content_release_failure_proportion": 0.1,
    "joint_failed_release_proportion": 0.01
  },
  {
    "gate": "plus_900_without_protected_content",
    "input_records": 1000,
    "released_records": 1000,
    "protected_content_released_records": 100,
    "failed_releases": 10,
    "coverage": 1.0,
    "protected_content_prevalence_among_releases": 0.1,
    "all_release_failure_proportion": 0.01,
    "protected_content_release_failure_proportion": 0.1,
    "joint_failed_release_proportion": 0.01
  }
]


### 2c. The keep/drop decomposition identifies the required comparison

For nested gates with unchanged outputs, let $\rho=\Pr(G_{\rm new}=1\mid G_{\rm old}=1)$, and let $r_{\rm keep}$ and $r_{\rm drop}$ denote the risks among old releases retained and removed by the new gate. For $0<\rho<1$,
\[
r_{\rm old}=\rho r_{\rm keep}+(1-\rho)r_{\rm drop},\qquad
r_{\rm new}-r_{\rm old}=(1-\rho)(r_{\rm keep}-r_{\rm drop}).
\]
Thus tighter selection decreases conditional risk exactly when retained cases are no riskier than dropped cases. In the 800-to-80 release example, all eight failures are retained, so $r_{\rm keep}=10\%$, $r_{\rm drop}=0$, and $\rho=0.1$. The old 1% risk is the weighted average; the new risk is 10%.

In [6]:
kept_ids = old_release_ids & new_release_ids
dropped_ids = old_release_ids - new_release_ids
rho_exact = len(kept_ids)/len(old_release_ids)
r_old_exact = len(unsafe_ids & old_release_ids)/len(old_release_ids)
r_keep_exact = len(unsafe_ids & kept_ids)/len(kept_ids)
r_drop_exact = len(unsafe_ids & dropped_ids)/len(dropped_ids)
assert kept_ids.isdisjoint(dropped_ids)
assert kept_ids | dropped_ids == old_release_ids
assert math.isclose(r_old_exact, rho_exact*r_keep_exact + (1-rho_exact)*r_drop_exact)
assert math.isclose(r_keep_exact-r_old_exact, (1-rho_exact)*(r_keep_exact-r_drop_exact))
keep_drop = {
    "old_released_records": len(old_release_ids), "kept_records": len(kept_ids),
    "dropped_records": len(dropped_ids),
    "kept_failures": len(unsafe_ids & kept_ids), "dropped_failures": len(unsafe_ids & dropped_ids),
    "retention_fraction_rho": rho_exact, "old_risk": r_old_exact,
    "keep_risk_new_risk": r_keep_exact, "drop_risk": r_drop_exact,
    "new_minus_old_risk": r_keep_exact-r_old_exact,
}
write_csv("gate_keep_drop.csv", [keep_drop])
print(json.dumps(keep_drop, indent=2))


{
  "old_released_records": 800,
  "kept_records": 80,
  "dropped_records": 720,
  "kept_failures": 8,
  "dropped_failures": 0,
  "retention_fraction_rho": 0.1,
  "old_risk": 0.01,
  "keep_risk_new_risk": 0.1,
  "drop_risk": 0.0,
  "new_minus_old_risk": 0.09000000000000001
}


### 2d. When threshold tightening is monotone, and when calibration is insufficient

A counterexample to arbitrary gate restriction does not imply that sensible thresholding must fail. For a fixed scalar score $S$, if the conditional failure function $\eta(s)=\Pr(L=1\mid S=s)$ is nondecreasing, then tightening a lower-score acceptance rule $G_t=\mathbf{1}\{S\le t\}$ cannot increase risk, provided the retained set has positive probability. This follows because the added high-score stratum has risk at least that of the retained low-score stratum. The next exact population is calibrated at every score bin and illustrates this monotonicity. Calibration here concerns **whole-record failure**, not mention confidence.

Calibration does not extend to every additional restriction. In a second population, every record has the constant score 0.01. Ten of 1,000 records fail, so the score is perfectly calibrated in the whole population. All ten failures lie in a subgroup of 100 records. Restricting release to that subgroup produces 10% conditional risk. This gate uses additional group information and is not merely a tighter threshold on the same score. No claim is made that this pattern is typical of real score models.

In [7]:
score_bins = [(0.01, 1000, 10), (0.05, 1000, 50),
              (0.10, 1000, 100), (0.20, 1000, 200)]
assert all(math.isclose(k/n, s) for s,n,k in score_bins)
monotonic_rows = []
for threshold in [.01, .05, .10, .20]:
    selected = [(s,n,k) for s,n,k in score_bins if s <= threshold]
    n = sum(item[1] for item in selected)
    k = sum(item[2] for item in selected)
    risk = k/n
    assert risk <= threshold + 1e-15
    monotonic_rows.append({"score_threshold": threshold, "released_records": n,
                           "failed_releases": k, "coverage": n/4000,
                           "conditional_failure_proportion": risk})
assert all(a["conditional_failure_proportion"] <= b["conditional_failure_proportion"]
           for a,b in zip(monotonic_rows, monotonic_rows[1:]))
write_csv("monotone_score_thresholds.csv", monotonic_rows)

constant_score_rows = [
    {"release_rule": "all", "constant_score": .01, "released_records": 1000,
     "failed_releases": 10, "conditional_failure_proportion": .01},
    {"release_rule": "restrict_to_group_A", "constant_score": .01, "released_records": 100,
     "failed_releases": 10, "conditional_failure_proportion": .10},
    {"release_rule": "group_B_for_diagnosis", "constant_score": .01, "released_records": 900,
     "failed_releases": 0, "conditional_failure_proportion": 0},
]
assert constant_score_rows[0]["constant_score"] == constant_score_rows[0]["conditional_failure_proportion"]
assert math.isclose(.1*constant_score_rows[1]["conditional_failure_proportion"] +
                    .9*constant_score_rows[2]["conditional_failure_proportion"], .01)
write_csv("calibration_group_restriction.csv", constant_score_rows)
print("Same-score thresholds:", json.dumps(monotonic_rows, indent=2))
print("Calibrated constant-score group restriction:", json.dumps(constant_score_rows, indent=2))


Same-score thresholds: [
  {
    "score_threshold": 0.01,
    "released_records": 1000,
    "failed_releases": 10,
    "coverage": 0.25,
    "conditional_failure_proportion": 0.01
  },
  {
    "score_threshold": 0.05,
    "released_records": 2000,
    "failed_releases": 60,
    "coverage": 0.5,
    "conditional_failure_proportion": 0.03
  },
  {
    "score_threshold": 0.1,
    "released_records": 3000,
    "failed_releases": 160,
    "coverage": 0.75,
    "conditional_failure_proportion": 0.05333333333333334
  },
  {
    "score_threshold": 0.2,
    "released_records": 4000,
    "failed_releases": 360,
    "coverage": 1.0,
    "conditional_failure_proportion": 0.09
  }
]
Calibrated constant-score group restriction: [
  {
    "release_rule": "all",
    "constant_score": 0.01,
    "released_records": 1000,
    "failed_releases": 10,
    "conditional_failure_proportion": 0.01
  },
  {
    "release_rule": "restrict_to_group_A",
    "constant_score": 0.01,
    "released_records": 100,
    "f

## 3. Zero-failure audit bounds and multiplicity

Assume a fixed complete release pipeline, a fixed sensitive-content policy, correct binary record-level audit labels, and $n$ independent records sampled from the pipeline's release-conditional deployment distribution. When all $n$ are safe, the exact one-sided Clopper-Pearson upper bound is
\[
U(n,\delta)=1-\delta^{1/n}.
\]
Here $1-\delta$ is frequentist confidence; it is not a posterior probability that the unknown risk is below the bound. At least
\[
n_{\min}=\left\lceil\frac{\log\delta}{\log(1-\alpha)}\right\rceil
\]
zero-failure audits are needed for $U\leq\alpha$.

For $H$ prespecified claims, allocating $\delta=0.05/H$ to each yields at least 95% simultaneous coverage by the union bound. Independence across the $H$ claims is not needed for this correction, but each claim must have a valid marginal audit design. This calculation does not authorize arbitrary adaptive reuse of an audit set. Counts in the table are **per claim**, and every bound assumes zero observed bad releases for that claim. A fixed family may contain correlated pipelines or groups. If the same records are eligible for multiple claims, the number of unique audited records need not be $Hn_{\min}$.

These binomial calculations use an independent-sampling model and are distinct from the preceding deterministic finite-population tables. Distribution shift, dependence, incomplete audit labels, and missed policy-sensitive content are not repaired by increasing the nominal confidence level.

In [8]:
def zero_failure_upper(n, delta=0.05):
    assert n >= 1 and 0 < delta < 1
    return -math.expm1(math.log(delta)/n)

def required_zero_failure_audits(alpha, delta=0.05):
    assert 0 < alpha < 1 and 0 < delta < 1
    n = math.ceil(math.log(delta)/math.log1p(-alpha))
    assert zero_failure_upper(n, delta) <= alpha
    assert n == 1 or zero_failure_upper(n-1, delta) > alpha
    return n

audit_rows = []
for alpha in [0.01, 0.001, 0.0001]:
    for H in [1, 20, 100]:
        delta = 0.05/H
        n = required_zero_failure_audits(alpha, delta)
        audit_rows.append({
            "target_risk_alpha": alpha, "prespecified_claims_H": H,
            "familywise_delta": 0.05, "per_claim_delta": delta,
            "zero_failure_audits_per_claim": n,
            "upper_bound_at_n": zero_failure_upper(n, delta),
            "upper_bound_at_n_minus_1": zero_failure_upper(n-1, delta),
        })
        # Cross-check the analytical result against the beta quantile form.
        assert math.isclose(zero_failure_upper(n, delta),
                            beta_distribution.ppf(1-delta, 1, n), rel_tol=2e-13)
write_csv("audit_sample_sizes.csv", audit_rows)

audit_comparison = []
for n in [100, 500]:
    for H in [1, 20, 100]:
        delta = 0.05/H
        audit_comparison.append({"zero_failure_audits": n, "prespecified_claims_H": H,
                                 "per_claim_delta": delta,
                                 "one_sided_upper_bound": zero_failure_upper(n, delta)})
write_csv("audit_100_vs_500.csv", audit_comparison)
for row in audit_rows:
    print(f"alpha={row['target_risk_alpha']:.4g}, H={row['prespecified_claims_H']:3d}, "
          f"n={row['zero_failure_audits_per_claim']:,}")
for row in audit_comparison:
    print(f"n={row['zero_failure_audits']:3d}, H={row['prespecified_claims_H']:3d}, "
          f"U={100*row['one_sided_upper_bound']:.6f}%")


alpha=0.01, H=  1, n=299
alpha=0.01, H= 20, n=597
alpha=0.01, H=100, n=757
alpha=0.001, H=  1, n=2,995
alpha=0.001, H= 20, n=5,989
alpha=0.001, H=100, n=7,598
alpha=0.0001, H=  1, n=29,956
alpha=0.0001, H= 20, n=59,912
alpha=0.0001, H=100, n=76,006
n=100, H=  1, U=2.951305%
n=100, H= 20, U=5.815508%
n=100, H=100, U=7.319216%
n=500, H=  1, U=0.597355%
n=500, H= 20, U=1.191142%
n=500, H=100, U=1.508684%


### 3a. A complete hypothetical audit readout

**The following counts are stipulated, not observed. No model or dataset audit was performed.** Suppose a complete plain-text release pipeline, protection policy, gate, and sampling plan are frozen before an independent audit. A release unit is a complete record; the recipient-visible object is the final redacted text. Assume independent input units from the claimed population, reliable complete policy labels, and no unaccounted subject dependence or adaptive stopping. Among $N=5,000$ inputs, $n=3,000$ are released and none fails containment. Of the released records, only $n_+=300$ contain policy-required content, and none of those fails.

The empirical coverage is 60%, the protected-content prevalence among releases is 10%, and both empirical failure proportions are zero. A **single** 95% upper bound for all released records is approximately 0.0998%, just below a 0.1% target. The separate 95% upper bound among released records with protected content is approximately 0.9936%, so the same target is unsupported there. These two marginal 95% statements are not a joint 95% statement. For at least 95% simultaneous coverage of both claims, allocate $\delta=0.025$ to each; their bounds become approximately 0.1229% and 1.2221%, respectively. Neither then supports the 0.1% target.

The 300 positive records are a subset, not 300 additional observations. Conditional on their count, a binomial calculation for $r_+$ uses that subset only. No confidence bound is claimed for the observed 10% prevalence, and it is not substituted into a population transfer formula as though known exactly. Real corpus size and provenance must independently justify the stated audit assumptions.

In [9]:
hypothetical_audit = {
    "status": "stipulated_counts_not_observations",
    "input_records": 5000, "released_records": 3000, "failed_releases": 0,
    "protected_content_released_records": 300, "protected_content_failed_releases": 0,
    "empirical_coverage": 3000/5000,
    "empirical_protected_content_prevalence_among_releases": 300/3000,
    "empirical_all_release_failure_proportion": 0.0,
    "empirical_protected_content_release_failure_proportion": 0.0,
    "target_risk": .001,
}
hypothetical_audit_rows = []
for mode,delta in [("individual_95_percent", .05), ("simultaneous_95_percent_two_claims", .025)]:
    for population,n in [("all_released_records", 3000),
                         ("released_records_with_protected_content", 300)]:
        upper = zero_failure_upper(n,delta)
        assert math.isclose(upper, beta_distribution.ppf(1-delta,1,n), rel_tol=2e-13)
        hypothetical_audit_rows.append({
            "status": "stipulated_counts_not_observations", "confidence_design": mode,
            "population": population, "input_records": 5000,
            "applicable_released_records": n, "failures": 0,
            "per_claim_delta": delta, "upper_risk_bound": upper,
            "supports_point_one_percent_target": upper <= .001,
        })
assert hypothetical_audit_rows[0]["supports_point_one_percent_target"]
assert not any(row["supports_point_one_percent_target"] for row in hypothetical_audit_rows[1:])
write_csv("hypothetical_audit_claim.csv", hypothetical_audit_rows)
for row in hypothetical_audit_rows:
    print(f"{row['confidence_design']}; {row['population']}; n={row['applicable_released_records']}; "
          f"upper={100*row['upper_risk_bound']:.6f}%")


individual_95_percent; all_released_records; n=3000; upper=0.099808%
individual_95_percent; released_records_with_protected_content; n=300; upper=0.993608%
simultaneous_95_percent_two_claims; all_released_records; n=3000; upper=0.122887%
simultaneous_95_percent_two_claims; released_records_with_protected_content; n=300; upper=1.222097%


### 3b. Gate-retention transfer needs jointly valid bounds

Consider a **separate hypothetical audit** of 3,000 independent releases from a frozen old mechanism, with zero failures. A frozen new gate retains 1,500 of those records. Assume the new gate is a subset of the old gate on the target population, the population is unchanged, and the specified new loss is pointwise no greater than the old loss. Then
\[
r_{\rm new}\leq\min\{1,r_{\rm old}/\rho\},\qquad
\rho=\Pr(G_{\rm new}=1\mid G_{\rm old}=1).
\]
The observed retention fraction 0.5 is not the population value of $\rho$. Allocate a total error budget of 0.05 as 0.025 for an upper bound $U_r$ on old risk and 0.025 for a lower bound $L_\rho$ on retention. For $s=1,500$ retained cases out of $m=3,000$,
\[
L_\rho=\operatorname{Beta}^{-1}(0.025;s,m-s+1).
\]
The two events hold jointly with probability at least 0.95 by the union bound, even though the estimates use the same records. On that event, $r_{\rm new}\leq\min\{1,U_r/L_\rho\}$. If the lower bound were zero, the conservative transferred upper bound would be one. This is a sufficient transfer bound, not a generally tight estimate of new risk. It requires gates fixed independently of the audit; arbitrary post-audit gate search is not covered. Transfer is useful when an aggregate old-risk bound and valid retention evidence are available but losses linked to the retained units are unavailable. When those labels are available and the outputs are unchanged, a direct bound can instead be recomputed from the existing retained audit labels; this does not require a fresh audit or relabeling.

In [10]:
def clopper_pearson_lower(successes, trials, delta):
    assert 0 <= successes <= trials and trials >= 1 and 0 < delta < 1
    return 0.0 if successes == 0 else float(beta_distribution.ppf(delta, successes, trials-successes+1))

old_audit_n, old_audit_k = 3000, 0
retained_audit_n = 1500
total_delta = .05
risk_delta = retention_delta = total_delta/2
old_risk_upper = zero_failure_upper(old_audit_n,risk_delta)
retention_lower = clopper_pearson_lower(retained_audit_n,old_audit_n,retention_delta)
retention_point_estimate = retained_audit_n/old_audit_n
transferred_upper = min(1.0, old_risk_upper/retention_lower) if retention_lower > 0 else 1.0
assert risk_delta+retention_delta == total_delta
assert 0 < retention_lower < retention_point_estimate
assert transferred_upper > old_risk_upper/retention_point_estimate
assert math.isclose(beta_distribution.cdf(retention_lower,1500,1501),.025,rel_tol=1e-10)
transfer_joint = {
    "status": "stipulated_counts_not_observations",
    "old_released_audit_records": old_audit_n, "old_failures": old_audit_k,
    "retained_records": retained_audit_n, "retention_point_estimate": retention_point_estimate,
    "total_delta": total_delta, "old_risk_delta": risk_delta, "retention_delta": retention_delta,
    "old_risk_upper_bound": old_risk_upper,
    "retention_lower_bound": retention_lower,
    "jointly_valid_transferred_risk_upper_bound": transferred_upper,
}
write_csv("gate_transfer_joint_bound.csv", [transfer_joint])
print(json.dumps(transfer_joint,indent=2))


{
  "status": "stipulated_counts_not_observations",
  "old_released_audit_records": 3000,
  "old_failures": 0,
  "retained_records": 1500,
  "retention_point_estimate": 0.5,
  "total_delta": 0.05,
  "old_risk_delta": 0.025,
  "retention_delta": 0.025,
  "old_risk_upper_bound": 0.001228870803825512,
  "retention_lower_bound": 0.48194876932698494,
  "jointly_valid_transferred_risk_upper_bound": 0.0025497955011723815
}


### 3c. An explicit release wrapper and a hypothetical audit-to-update decision

This example specifies a release wrapper around a mathematically fixed deterministic span extractor $f$. **No trained model or weights are selected, no real extractor is evaluated, and the counts below remain stipulated.** The extractor is a parameter of the mechanism, fixed independently of the audit. The example therefore illustrates what an audit statement and an update decision would mean, not an actual model certification.

The baseline gate $g_0$ accepts only a valid UTF-8 input of at most 4,096 Unicode code points for which extraction, validation, and rendering succeed. The extractor returns a finite list of half-open intervals with integer offsets, labels in `PERSON`, `EMAIL`, or `PHONE`, and finite scores in [0,1]. Offsets must satisfy $0\leq a<b\leq |x|$. An empty list and overlapping valid intervals are allowed. All returned items must validate, including those below the masking threshold. The union of intervals with score at least 0.5 is concealed by replacing each covered Unicode code point with one literal ASCII `*`. The only recipient-visible object is that final text; original text, offsets, scores, and mask traces are internal and are not appended to the released artifact. Input length and span offsets refer to code points, not UTF-8 bytes.

Update 1, the positive control, expands every original mask interval by one code point at each end, clipped to the source boundaries. It takes the union with the original mask and reuses **exactly the original $g_0$ decision**. This operation is total on the validated Unicode strings. There is no new acceptance test or extractor call in the positive control. Any actual implementation that changed the gate or exposed another representation would need its own justification.

Update 2, a separate prespecified restriction, limits release to $g_2(x)=g_0(x)\mathbf{1}\{|x|\leq2,048\}$ while retaining the baseline output. The length threshold is fixed before the audit, not chosen after examining failures. A hypothetical audit has 5,000 independent input units, 3,000 baseline releases, zero baseline failures, and 300 baseline releases containing policy-required content. The restricted gate retains 1,500 of the 3,000 releases. No count of protected-content releases under the restricted gate is stipulated, so no such subgroup claim is inferred.

At individual 95% confidence, the baseline bound is 0.0998079%, supporting a 0.1% target under the stated assumptions. Mask expansion inherits that same event and bound. For the restricted gate, the previously calculated joint transfer procedure allocates 0.025 to the old-risk upper bound and 0.025 to the retention lower bound, giving a 0.2549796% upper bound. Recomputing a separate individual 95% bound from the existing audit labels of the 1,500 retained units gives approximately 0.1995162%. This uses the original labels and unchanged outputs, not a fresh audit or relabeling. Neither procedure establishes the 0.1% target for the restricted gate. The smaller retained sample leaves that target unsupported even though it still contains zero failures. **This calculation demonstrates neither an increase in true risk nor a violation of the target.**

The rows compare distinct confidence procedures, not a family of simultaneously 95%-valid statements. The baseline and mask-expansion rows share one event. The transfer row has its own joint 95% event, and the direct restricted-gate row is a separate individual 95% procedure. The earlier simultaneous all-release/protected-content example is another explicitly separate confidence plan.

A single valid event $\{r_0\leq U_0\}$ supports every update proved to have no greater loss pointwise with the same policy, population, and gate. It is unnecessary to allocate additional error probability to each structurally dominated update, even for many such updates, because all their guarantees follow on the same event. This requires a valid structural argument for every update, not selecting apparently favorable empirical results and declaring dominance.

In [11]:
ALLOWED_LABELS = frozenset({"PERSON", "EMAIL", "PHONE"})
REPLACEMENT = "*"

def withheld(reason):
    return {"released": False, "source": None, "text": None, "mask": [], "reason": reason}

def baseline_wrapper(raw, extractor):
    """Explicit wrapper around a fixed extractor parameter; no real model is supplied."""
    try:
        if isinstance(raw, bytes):
            source = raw.decode("utf-8", errors="strict")
        elif isinstance(raw, str):
            raw.encode("utf-8", errors="strict")
            source = raw
        else:
            return withheld("input_type")
        if len(source) > 4096:
            return withheld("length")
        predictions = extractor(source)
        if not isinstance(predictions, (list, tuple)):
            return withheld("prediction_container")
        intervals = []
        for prediction in predictions:
            if not isinstance(prediction, (list, tuple)) or len(prediction) != 4:
                return withheld("prediction_shape")
            start,end,label,score = prediction
            if (type(start) is not int or type(end) is not int or
                not 0 <= start < end <= len(source)):
                return withheld("offset")
            if not isinstance(label,str) or label not in ALLOWED_LABELS:
                return withheld("label")
            if (isinstance(score,bool) or not isinstance(score,(int,float)) or
                not math.isfinite(score) or not 0 <= score <= 1):
                return withheld("score")
            if score >= .5:
                intervals.append((start,end))
        mask = [False]*len(source)
        for start,end in intervals:
            mask[start:end] = [True]*(end-start)
        rendered = "".join(REPLACEMENT if hidden else character
                           for character,hidden in zip(source,mask))
        rendered.encode("utf-8", errors="strict")
        if len(rendered) != len(source):
            return withheld("rendering")
        return {"released": True, "source": source, "text": rendered,
                "mask": mask, "reason": "accepted"}
    except Exception:
        return withheld("extraction_validation_or_rendering_exception")

def expanded_mask_wrapper(baseline):
    """Total mask dilation on validated inputs; the original gate is reused."""
    if not baseline["released"]:
        return baseline
    source = baseline["source"]
    old_mask = baseline["mask"]
    expanded = old_mask.copy()
    for index,hidden in enumerate(old_mask):
        if hidden:
            for adjacent in range(max(0,index-1),min(len(source),index+2)):
                expanded[adjacent] = True
    result = dict(baseline)
    result["mask"] = expanded
    result["text"] = "".join(REPLACEMENT if hidden else character
                               for character,hidden in zip(source,expanded))
    return result

def restricted_gate_wrapper(baseline):
    if not baseline["released"] or len(baseline["source"]) <= 2048:
        return baseline
    return withheld("prespecified_2048_codepoint_gate")

# Handcrafted software fixtures only. These are not the hypothetical audit units.
fixture_text = "*Aé🙂B mail@example.org"
fixture_predictions = {
    fixture_text: [(2,4,"PERSON",.5), (3,4,"PERSON",.8),
                   (4,5,"PHONE",.49), (6,len(fixture_text),"EMAIL",.9)],
    "bad label": [(0,3,"OTHER",.9)],
    "bad score": [(0,3,"PERSON",float("nan"))],
    "bad offset": [(0,100,"PERSON",.9)],
    "bad integer": [(False,3,"PERSON",.9)],
}
def fixed_fixture_extractor(source):
    return fixture_predictions.get(source,[])

fixture_base = baseline_wrapper(fixture_text,fixed_fixture_extractor)
fixture_expanded = expanded_mask_wrapper(fixture_base)
assert fixture_base["released"] and fixture_expanded["released"]
assert fixture_base["text"] == "*A**B " + "*"*len("mail@example.org")
assert len(fixture_base["text"]) == len(fixture_text)
assert len(fixture_text.encode("utf-8")) > len(fixture_text)  # Distinct byte/code-point lengths.
assert not fixture_base["mask"][0]  # An original '*' is not evidence of a mask.
assert all(not old or new for old,new in zip(fixture_base["mask"],fixture_expanded["mask"]))
assert fixture_expanded["text"] == "*"*len(fixture_text)
assert not fixture_expanded["mask"][0]  # Internal mask provenance remains distinct from the text.
assert restricted_gate_wrapper(fixture_base)["released"]

fixture_inputs = [fixture_text, "", "plain text", "x"*3000, "x"*4097,
                  "bad label", "bad score", "bad offset", "bad integer", b"\xff", "\ud800"]
for raw in fixture_inputs:
    base = baseline_wrapper(raw,fixed_fixture_extractor)
    expanded = expanded_mask_wrapper(base)
    assert expanded["released"] == base["released"]
    if base["released"]:
        assert len(expanded["text"]) == len(base["source"])
        assert all(not old or new for old,new in zip(base["mask"],expanded["mask"]))
assert baseline_wrapper("",fixed_fixture_extractor)["released"]
assert baseline_wrapper("x"*3000,fixed_fixture_extractor)["released"]
assert not restricted_gate_wrapper(baseline_wrapper("x"*3000,fixed_fixture_extractor))["released"]
assert all(not baseline_wrapper(raw,fixed_fixture_extractor)["released"]
           for raw in ["x"*4097,"bad label","bad score","bad offset","bad integer",b"\xff","\ud800"])
wrapper_verification = {"status":"handcrafted_software_checks_only_not_an_audit",
                        "fixed_extractor_is_a_mathematical_parameter":True,
                        "real_model_evaluated":False,
                        "replacement_character":REPLACEMENT,
                        "all_handcrafted_checks_passed":True}
print("Handcrafted wrapper checks passed; no empirical audit was performed.")


Handcrafted wrapper checks passed; no empirical audit was performed.


In [12]:
update_target = .001
baseline_individual_upper = zero_failure_upper(3000,.05)
direct_restricted_upper = zero_failure_upper(1500,.05)
update_decisions = []
for name,rule,n,bound,plan,delta_r,delta_retention,decision in [
    ("baseline", "g0: validated UTF-8 and length <= 4096", 3000,
     baseline_individual_upper, "individual_baseline_95_percent", .05, "",
     "target_established_under_hypothetical_assumptions"),
    ("mask_expansion", "same g0; expand masks one code point at both ends", 3000,
     baseline_individual_upper, "inherits_the_baseline_confidence_event", .05, "",
     "target_inherited_by_pointwise_loss_dominance"),
    ("restricted_gate_transfer", "g2 = g0 and length <= 2048", 1500,
     transfer_joint["jointly_valid_transferred_risk_upper_bound"],
     "joint_transfer_plan_95_percent", .025, .025,
     "target_not_established_not_proved_violated"),
    ("restricted_gate_direct", "g2 = g0 and length <= 2048", 1500,
     direct_restricted_upper, "separate_individual_direct_95_percent", .05, "",
     "target_not_established_not_proved_violated"),
]:
    update_decisions.append({
        "status":"stipulated_audit_counts_not_pipeline_measurements",
        "update":name, "release_rule":rule, "input_records":5000,
        "applicable_released_records":n, "failures":0,
        "protected_content_released_records":300 if n == 3000 else "not_stipulated",
        "confidence_procedure":plan, "risk_delta":delta_r,
        "retention_delta":delta_retention,
        "retention_lower_bound":transfer_joint["retention_lower_bound"] if name == "restricted_gate_transfer" else "not_used",
        "upper_risk_bound":bound, "target_risk":update_target,
        "target_established":bound <= update_target,
        "decision":decision,
        "simultaneous_95_percent_coverage_across_all_rows_claimed":False,
    })
assert baseline_individual_upper <= update_target
assert direct_restricted_upper > update_target
assert update_decisions[0]["upper_risk_bound"] == update_decisions[1]["upper_risk_bound"]
assert [row["target_established"] for row in update_decisions] == [True,True,False,False]
assert math.isclose(direct_restricted_upper,beta_distribution.ppf(.95,1,1500),rel_tol=2e-13)
write_csv("update_decisions.csv",update_decisions)
for row in update_decisions:
    print(f"{row['update']}: upper={100*row['upper_risk_bound']:.7f}%; {row['decision']}")


baseline: upper=0.0998079%; target_established_under_hypothetical_assumptions
mask_expansion: upper=0.0998079%; target_inherited_by_pointwise_loss_dominance
restricted_gate_transfer: upper=0.2549796%; target_not_established_not_proved_violated
restricted_gate_direct: upper=0.1995162%; target_not_established_not_proved_violated


## 4. A candidate-only certificate can have an omission floor

Consider a pipeline that can redact only regions in its generated candidate set. All 100,000 constructed records are released. In 200 of them, at least one policy-sensitive span is outside the union of every candidate region that can be redacted. A perfect candidate classifier removes all available sensitive candidates. It still cannot remove those 200 omitted spans. Every other record is safe.

For this **candidate-restricted pipeline with no additional suppression or fallback**, the bad-release rate is exactly $200/100{,}000=0.002$. More generally, if $O$ is the event that a released record contains a sensitive span outside the union of all possible candidate-based masks, then $O\subseteq B$ and
\[
\Pr(B\mid G=1)\geq\Pr(O\mid G=1).
\]
Thus a target of 0.1% is unattainable in the constructed population under those restrictions, even with zero candidate-classification errors. The floor can change if the system expands its mask set or withholds records. No universal floor is asserted for pipelines capable of suppressing whole records.

A full-record audit would count omitted spans as failures. Candidate-only auditing cannot estimate this floor unless its sampling and labels also inspect content outside the candidate set. This is a limitation of the audited event, not a failure of the Clopper-Pearson formula.

In [13]:
candidate_example = {
    "released_records": 100_000,
    "records_with_sensitive_content_outside_all_candidate_masks": 200,
    "candidate_classification_errors": 0,
    "additional_suppression_or_fallback": False,
    "actual_bad_release_rate": 200/100_000,
    "omission_floor": 200/100_000,
    "target_risk_alpha": 0.001,
    "target_attainable_under_stated_restrictions": False,
}
assert candidate_example["actual_bad_release_rate"] == candidate_example["omission_floor"]
assert candidate_example["omission_floor"] > candidate_example["target_risk_alpha"]
write_csv("candidate_omission_floor.csv", [candidate_example])
print(json.dumps(candidate_example, indent=2))


{
  "released_records": 100000,
  "records_with_sensitive_content_outside_all_candidate_masks": 200,
  "candidate_classification_errors": 0,
  "additional_suppression_or_fallback": false,
  "actual_bad_release_rate": 0.002,
  "omission_floor": 0.002,
  "target_risk_alpha": 0.001,
  "target_attainable_under_stated_restrictions": false
}


### 4a. Rectangle matching and complete concealment are different losses

These are **exact hypothetical geometries**, not image-model outputs. Let the single required gold rectangle have area $a$. A matching opaque mask has intersection-over-union (IoU) one and conceals the whole gold region. A containing expanded mask of area $2a$ still conceals the whole region, but has IoU 0.5 and fails an IoU threshold of 0.75. Conversely, an inner mask of area $0.8a$ has IoU 0.8 and passes that matching threshold while leaving one fifth of the required region uncovered.

Each case has one gold rectangle, one mask, and the same correct type label, so neither label disagreement nor one-to-one assignment explains the difference. These geometries show why increasing concealment need not improve a localization-matching score. They do not equate every uncovered image pixel with a successful inference attack.

In [14]:
def rectangle_area(rect):
    x0,y0,x1,y1 = rect
    assert x1 >= x0 and y1 >= y0
    return (x1-x0)*(y1-y0)

def rectangle_intersection_area(first,second):
    return max(0,min(first[2],second[2])-max(first[0],second[0])) * \
           max(0,min(first[3],second[3])-max(first[1],second[1]))

def rectangle_contains(outer,inner):
    return (outer[0] <= inner[0] and outer[1] <= inner[1] and
            outer[2] >= inner[2] and outer[3] >= inner[3])

gold_rectangle = (0,0,10,10)
gold_area = rectangle_area(gold_rectangle)
geometry_rows = []
for name,mask in [("matching",(0,0,10,10)),
                  ("containing_expansion",(0,0,20,10)),
                  ("inner_partial_mask",(0,0,8,10))]:
    area = rectangle_area(mask)
    intersection = rectangle_intersection_area(gold_rectangle,mask)
    iou = intersection/(gold_area+area-intersection)
    geometry_rows.append({
        "scenario": name, "status": "exact_hypothetical_geometry",
        "gold_area": gold_area, "opaque_mask_area": area,
        "mask_area_over_gold_area": area/gold_area,
        "intersection_area": intersection, "intersection_over_union": iou,
        "matching_threshold": .75, "same_correct_type": True,
        "passes_iou_matching": iou >= .75,
        "complete_concealment": rectangle_contains(mask,gold_rectangle),
        "uncovered_gold_fraction": 1-intersection/gold_area,
    })
assert [row["passes_iou_matching"] for row in geometry_rows] == [True,False,True]
assert [row["complete_concealment"] for row in geometry_rows] == [True,True,False]
assert all(math.isclose(row["intersection_over_union"],value)
           for row,value in zip(geometry_rows,[1,.5,.8]))
assert math.isclose(geometry_rows[2]["uncovered_gold_fraction"],.2)
write_csv("geometry_containment_vs_matching.csv",geometry_rows)
print(json.dumps(geometry_rows,indent=2))


[
  {
    "scenario": "matching",
    "status": "exact_hypothetical_geometry",
    "gold_area": 100,
    "opaque_mask_area": 100,
    "mask_area_over_gold_area": 1.0,
    "intersection_area": 100,
    "intersection_over_union": 1.0,
    "matching_threshold": 0.75,
    "same_correct_type": true,
    "passes_iou_matching": true,
    "complete_concealment": true,
    "uncovered_gold_fraction": 0.0
  },
  {
    "scenario": "containing_expansion",
    "status": "exact_hypothetical_geometry",
    "gold_area": 100,
    "opaque_mask_area": 200,
    "mask_area_over_gold_area": 2.0,
    "intersection_area": 100,
    "intersection_over_union": 0.5,
    "matching_threshold": 0.75,
    "same_correct_type": true,
    "passes_iou_matching": false,
    "complete_concealment": true,
    "uncovered_gold_fraction": 0.0
  },
  {
    "scenario": "inner_partial_mask",
    "status": "exact_hypothetical_geometry",
    "gold_area": 100,
    "opaque_mask_area": 80,
    "mask_area_over_gold_area": 0.8,
    "inte

### 4b. Withholding invalid outputs cannot worsen this matching loss

This is a restricted matching-protocol case, not a statement about arbitrary gate changes. Consider only applicable pages, each containing at least one required gold instance. If invalid outputs are assigned empty predictions, every invalid output fails the specified matching loss. Withholding exactly these invalid outputs removes only failures.

Let $d$ be the original failure proportion and $v_+>0$ the fraction of applicable pages with valid outputs. The failure proportion among valid outputs is
\[
d_{\rm valid}=\frac{d-(1-v_+)}{v_+}\leq d.
\]
For 20 hypothetical applicable pages, eight fail and five of those failures are invalid outputs. There are three failures among 15 valid outputs, so withholding the five invalid outputs reduces matching failure from 40% to 20%. The next cell also checks every feasible failure/invalid-count pair for a 20-page population. This monotonicity depends on all invalid outputs being failures under the same fixed loss; it does not extend to arbitrary routing decisions or inferential privacy.

In [15]:
applicable_pages, matching_failures, invalid_outputs = 20,8,5
valid_pages = applicable_pages-invalid_outputs
valid_failures = matching_failures-invalid_outputs
d = matching_failures/applicable_pages
v_positive = valid_pages/applicable_pages
d_valid = valid_failures/valid_pages
assert invalid_outputs <= matching_failures
assert math.isclose(d_valid,(d-(1-v_positive))/v_positive)
assert d_valid <= d
checked_states = 0
for failures in range(21):
    for invalid in range(failures+1):
        if invalid == 20:
            continue  # No valid outputs: the conditional quantity is undefined.
        valid_fraction = (20-invalid)/20
        before = failures/20
        after = (failures-invalid)/(20-invalid)
        assert math.isclose(after,(before-(1-valid_fraction))/valid_fraction,abs_tol=1e-14)
        assert after <= before+1e-14
        checked_states += 1
assert checked_states == 230
invalid_output_example = {
    "status": "exact_hypothetical_matching_protocol",
    "applicable_pages": applicable_pages, "original_matching_failures": matching_failures,
    "invalid_outputs_all_matching_failures": invalid_outputs,
    "valid_pages": valid_pages, "valid_output_matching_failures": valid_failures,
    "original_matching_failure_proportion": d,
    "valid_fraction_among_applicable_pages": v_positive,
    "valid_output_matching_failure_proportion": d_valid,
    "feasible_count_states_checked_at_N20": checked_states,
}
write_csv("invalid_output_withholding.csv",[invalid_output_example])
print(json.dumps(invalid_output_example,indent=2))


{
  "status": "exact_hypothetical_matching_protocol",
  "applicable_pages": 20,
  "original_matching_failures": 8,
  "invalid_outputs_all_matching_failures": 5,
  "valid_pages": 15,
  "valid_output_matching_failures": 3,
  "original_matching_failure_proportion": 0.4,
  "valid_fraction_among_applicable_pages": 0.75,
  "valid_output_matching_failure_proportion": 0.2,
  "feasible_count_states_checked_at_N20": 230
}


## 5. Publication figures

`figure3_gate_and_denominator.pdf` is the compact main-paper figure, showing two exact constructions: gate tightening and dilution by released records without protected content. The earlier mention-score and sample-size figures remain supplementary illustrations. PDF outputs contain vector graphics and embedded fonts; PNG copies support inspection. Captions must identify the constructions as hypothetical and the audit calculations as idealized, not observations or model experiments.

In [16]:
# Figure 1: distinct metrics and distinct denominators.
fig, axes = plt.subplots(1, 3, figsize=(7.16, 2.60), layout="constrained",
                         gridspec_kw={"width_ratios": [1.06, 1, 1]})
ax = axes[0]
labels = ["Recall", r"$F_1$", r"$F_{10}$"]
vals = [metric_rows[0][k] for k in ["micro_recall", "micro_F1", "micro_F10"]]
x = np.arange(len(labels))
ax.bar(x-.16, vals, width=.30, color=BLUE, label="System A")
ax.bar(x+.16, vals, width=.30, color=ORANGE, label="System B")
for xi, v in zip(x, vals):
    ax.text(xi, v+.015, f"{v:.4f}", ha="center", va="bottom", fontsize=7)
ax.set_xticks(x, labels)
ax.set_ylim(0, 1.16)
ax.set_yticks([0, .5, 1])
ax.set_ylabel("Micro mention score")
ax.set_title("(a) Identical mention scores", pad=8)
fig.legend(*ax.get_legend_handles_labels(), loc="outside lower center",
           ncols=2, frameon=False, handlelength=1, borderpad=.2)
ax.yaxis.grid(True, alpha=.25)
ax.set_axisbelow(True)

ax = axes[1]
ax.bar([0,1], [100,1], width=.56, color=[BLUE,ORANGE])
for xi,v in enumerate([100,1]):
    ax.text(xi,v+2.2,f"{v}%",ha="center",va="bottom",fontsize=8)
ax.set_xticks([0,1],["System A", "System B"])
ax.set_ylabel("Bad released records (%)")
ax.set_ylim(0,116)
ax.set_yticks([0,25,50,75,100])
ax.set_title("(b) Different record risks", pad=8)
ax.yaxis.grid(True, alpha=.25)
ax.set_axisbelow(True)

ax = axes[2]
ax.bar([0,1],[1,10],width=.56,color=["#727272",GREEN])
for xi,v in enumerate([1,10]):
    ax.text(xi,v+.25,f"{v}%",ha="center",va="bottom",fontsize=8)
ax.set_xticks([0,1],["All processed\nrecords", "Released\nrecords"])
ax.set_ylabel("Bad releases / denominator (%)")
ax.set_ylim(0,11.6)
ax.set_yticks([0,2,4,6,8,10])
ax.set_title("(c) Conditioning on release", pad=8)
ax.yaxis.grid(True,alpha=.25)
ax.set_axisbelow(True)
fig.savefig(OUT/"figure1_constructed_risks.pdf")
fig.savefig(OUT/"figure1_constructed_risks.png")
plt.close(fig)

# Figure 2: exact, one-sided upper bounds after zero observed failures.
fig, ax = plt.subplots(figsize=(3.50,2.72), layout="constrained")
ns = np.geomspace(50,200_000,500)
for H,color,style in [(1,BLUE,"-"),(20,ORANGE,"--"),(100,GREEN,"-.")]:
    bounds = -np.expm1(np.log(.05/H)/ns)
    ax.plot(ns,bounds*100,color=color,ls=style,lw=1.45,label=f"H = {H}")
for alpha,label in [(.01,"1%"),(.001,"0.1%"),(.0001,"0.01%")]:
    ax.axhline(alpha*100,color="#8A8A8A",lw=.6,zorder=0)
    ax.text(180_000,alpha*100*1.10,label,fontsize=7,color="#555555", ha="right")
ax.scatter([100,500],[100*zero_failure_upper(n) for n in [100,500]],
           s=18,facecolor="white",edgecolor=BLUE,zorder=4)
ax.annotate("100 audits: 2.95%",xy=(100,100*zero_failure_upper(100)),
            xytext=(270,12),fontsize=6.8,color=BLUE,
            arrowprops={"arrowstyle":"-","lw":.6,"color":BLUE})
ax.annotate("500 audits: 0.597%",xy=(500,100*zero_failure_upper(500)),
            xytext=(1100,2.1),fontsize=6.8,color=BLUE,
            arrowprops={"arrowstyle":"-","lw":.6,"color":BLUE})
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(50,200_000)
ax.set_ylim(.001,20)
ax.set_xlabel("Audited released records per claim, n")
ax.set_ylabel("Upper bound on bad-release risk (%)")
ax.set_xticks([100,1000,10000,100000],["100","1,000","10,000","100,000"])
ax.set_yticks([.001,.01,.1,1,10],["0.001","0.01","0.1","1","10"])
ax.legend(loc="lower left",frameon=False,title="Prespecified claims",title_fontsize=7,
          handlelength=2.4, borderaxespad=.5)
ax.grid(True,which="major",alpha=.16)
fig.savefig(OUT/"figure2_zero_failure_audits.pdf")
fig.savefig(OUT/"figure2_zero_failure_audits.png")
plt.close(fig)
print("Wrote two vector PDF figures and PNG inspection copies.")


Wrote two vector PDF figures and PNG inspection copies.


In [17]:
# Main-paper figure: exact gate changes and protected-content denominator dilution.
fig, axes = plt.subplots(1,2,figsize=(7.16,2.72),layout="constrained")
width = .32
x = np.array([0,1])
ax = axes[0]
conditional = [100*row["conditional_bad_release_rate"] for row in gate_tightening_rows]
joint = [100*row["joint_bad_and_released_rate"] for row in gate_tightening_rows]
ax.bar(x-width/2,conditional,width=width,color=BLUE,label="Failure among released records")
ax.bar(x+width/2,joint,width=width,color="#888888",label="Failed releases / all inputs")
for xx,yy in zip(x-width/2,conditional):
    ax.text(xx,yy+.23,f"{yy:g}%",ha="center",va="bottom",fontsize=8)
for xx,yy in zip(x+width/2,joint):
    ax.text(xx,yy+.23,f"{yy:g}%",ha="center",va="bottom",fontsize=8)
ax.set_xticks(x,["Old gate\n800 released", "Stricter gate\n80 released"])
ax.set_title("(a) Gate tightening: same eight failures",pad=8)
ax.set_ylabel("Failure proportion (%)")
ax.set_ylim(0,15)
ax.set_yticks([0,5,10])
ax.legend(loc="upper left",frameon=False,fontsize=7.1,handlelength=1)
ax.yaxis.grid(True,alpha=.20)
ax.set_axisbelow(True)

ax = axes[1]
all_release = [100*row["all_release_failure_proportion"] for row in dilution_rows]
positive_release = [100*row["protected_content_release_failure_proportion"] for row in dilution_rows]
ax.bar(x-width/2,all_release,width=width,color=BLUE,label="Failure among released records")
ax.bar(x+width/2,positive_release,width=width,color=ORANGE,label="Failure among protected-content releases")
for xx,yy in zip(x-width/2,all_release):
    ax.text(xx,yy+.23,f"{yy:g}%",ha="center",va="bottom",fontsize=8)
for xx,yy in zip(x+width/2,positive_release):
    ax.text(xx,yy+.23,f"{yy:g}%",ha="center",va="bottom",fontsize=8)
ax.set_xticks(x,["100 released\n100 with protected content",
                  "1,000 released\n100 with protected content"])
ax.set_title("(b) Denominator dilution: same ten failures",pad=8)
ax.set_ylabel("Failure proportion (%)")
ax.set_ylim(0,15)
ax.set_yticks([0,5,10])
ax.legend(loc="upper left",frameon=False,fontsize=7.1,handlelength=1)
ax.yaxis.grid(True,alpha=.20)
ax.set_axisbelow(True)
fig.savefig(OUT/"figure3_gate_and_denominator.pdf",metadata={"Author":"Anonymous Author"})
fig.savefig(OUT/"figure3_gate_and_denominator.png")
plt.close(fig)
print("Wrote main figure: figure3_gate_and_denominator.pdf (exact hypothetical populations).")


Wrote main figure: figure3_gate_and_denominator.pdf (exact hypothetical populations).


## 6. Machine-readable summary and verification

Every numerical result above is derived from an explicit finite construction or the stated binomial formula. In this notebook, no training, calibration, model selection, benchmark evaluation, or real-world audit has been performed. The final assertions only check the internal arithmetic and assumptions of these illustrations.

In [18]:
results = {
    "status": "analytical_illustrations_only_no_real_world_measurements",
    "manuscript_title": "Position: PII Redaction Claims Must Specify the Release Mechanism",
    "environment": VERSIONS,
    "mention_vs_record": metric_rows,
    "release_conditioning": release_illustration,
    "release_selection_sweep": selection_rows,
    "gate_tightening": gate_tightening_rows,
    "protected_content_denominator": dilution_rows,
    "keep_drop_identity": keep_drop,
    "monotone_score_thresholds": monotonic_rows,
    "calibration_group_restriction": constant_score_rows,
    "hypothetical_audit_counts": hypothetical_audit,
    "hypothetical_audit_bounds": hypothetical_audit_rows,
    "gate_transfer_joint_bound": transfer_joint,
    "audit_to_update_wrapper_verification": wrapper_verification,
    "audit_to_update_decisions": update_decisions,
    "audit_sample_sizes": audit_rows,
    "audit_100_vs_500": audit_comparison,
    "candidate_omission_example": candidate_example,
    "geometric_containment_vs_matching": geometry_rows,
    "invalid_output_withholding": invalid_output_example,
    "deterministic_assertions_passed": True,
}
(OUT/"results.json").write_text(json.dumps(results,indent=2,allow_nan=False)+"\n")
expected = ["environment.json", "results.json", "mention_vs_record.csv",
            "release_population.csv", "release_selection_sweep.csv", "gate_tightening.csv",
            "audit_sample_sizes.csv", "audit_100_vs_500.csv", "candidate_omission_floor.csv",
            "figure1_constructed_risks.pdf", "figure1_constructed_risks.png",
            "figure2_zero_failure_audits.pdf", "figure2_zero_failure_audits.png",
            "protected_content_denominator.csv", "gate_keep_drop.csv",
            "monotone_score_thresholds.csv", "calibration_group_restriction.csv",
            "hypothetical_audit_claim.csv", "gate_transfer_joint_bound.csv",
            "figure3_gate_and_denominator.pdf", "figure3_gate_and_denominator.png",
            "geometry_containment_vs_matching.csv", "invalid_output_withholding.csv",
            "update_decisions.csv"]
assert all((OUT/name).is_file() and (OUT/name).stat().st_size > 0 for name in expected)
print("All deterministic checks passed.")
print("Output directory:", OUT.name)
print("Files:", ", ".join(expected))


All deterministic checks passed.
Output directory: illustration_outputs
Files: environment.json, results.json, mention_vs_record.csv, release_population.csv, release_selection_sweep.csv, gate_tightening.csv, audit_sample_sizes.csv, audit_100_vs_500.csv, candidate_omission_floor.csv, figure1_constructed_risks.pdf, figure1_constructed_risks.png, figure2_zero_failure_audits.pdf, figure2_zero_failure_audits.png, protected_content_denominator.csv, gate_keep_drop.csv, monotone_score_thresholds.csv, calibration_group_restriction.csv, hypothetical_audit_claim.csv, gate_transfer_joint_bound.csv, figure3_gate_and_denominator.pdf, figure3_gate_and_denominator.png, geometry_containment_vs_matching.csv, invalid_output_withholding.csv, update_decisions.csv
